In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [9]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

import warnings
warnings.filterwarnings("ignore")

# Dagshub/MLflow Initialization

In [10]:
pip install dagshub mlflow scikit-learn pandas matplotlib seaborn skops --quiet

Note: you may need to restart the kernel to use updated packages.


In [11]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='Nestor-Dzadzamia', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)

Initialized MLflow to track repo "Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection"

Repository Nestor-Dzadzamia/IEEE-CIS-Fraud-Detection initialized!

# Data analysis

In [12]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

In [13]:
print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


In [14]:
print('transaction data --------------------------------------------------------------------')
print(train_transaction.head(10))

print('missing values -----------------------------------------------------------------------')
missing_values = train_transaction.isnull().mean().sort_values(ascending=False)
print(missing_values.head(20))

print('identity table -------------------------------------------------------------------------')
print(train_identity.head(10))
print('target distribution --------------------------------------------------------------------')
print(train_transaction['isFraud'].value_counts(normalize=True))

transaction data --------------------------------------------------------------------
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   
5        2987005        0          86510            49.0         W   5937   
6        2987006        0          86522           159.0         W  12308   
7        2987007        0          86529           422.5         W  12695   
8        2987008        0          86535            15.0         H   2803   
9        2987009        0          86536           117.0         W  17399   

   card2  card3       card4  card5   card6  addr1  addr2  dist1  d

# Merging data frames

In [15]:
df = train_transaction.merge(train_identity, on='TransactionID', how='left')
print(f"Merged shape: {df.shape}")

y = df['isFraud']
X = df.drop(columns=['isFraud', 'TransactionID'])

print(f"X shape: {X.shape}")
print(f"Fraud ratio: {y.mean():.4f}")

Merged shape: (590540, 434)
X shape: (590540, 432)
Fraud ratio: 0.0350


# train/test split


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Train fraud ratio: {y_train.mean():.4f}")
print(f"Test fraud ratio: {y_test.mean():.4f}")

X_train shape: (472432, 432)
X_test shape: (118108, 432)
Train fraud ratio: 0.0350
Test fraud ratio: 0.0350


# Creating Preprocessor class

In [17]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, null_thresh=0.5, corr_thresh=0.9):
        self.null_thresh = null_thresh
        self.corr_thresh = corr_thresh

    def _feature_engineering(self, X):
        X = X.copy()
        # Log of transaction amount
        X['log_TransactionAmt'] = np.log1p(X['TransactionAmt'])
        # Transaction hour and day
        X['Transaction_hour'] = (X['TransactionDT'] // 3600) % 24
        X['Transaction_day'] = (X['TransactionDT'] // (3600 * 24)) % 7
        return X

    def fit(self, X, y=None):
        X = self._feature_engineering(X)
        
        # Drop high null columns
        self.cols_to_drop_ = X.columns[X.isnull().mean() > self.null_thresh].tolist()
        X = X.drop(columns=self.cols_to_drop_)
        
        # Drop highly correlated features
        corr_matrix = X.select_dtypes(include=np.number).corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.corr_cols_to_drop_ = [col for col in upper.columns if any(upper[col] > self.corr_thresh)]
        X = X.drop(columns=self.corr_cols_to_drop_)
        
        # Categorical and numerical columns
        self.cat_cols_ = [col for col in X.columns if X[col].dtype == 'object']
        self.num_cols_ = [col for col in X.columns if X[col].dtype != 'object']
        
        # Fit imputers
        self.num_imputer_ = SimpleImputer(strategy='median')
        self.num_imputer_.fit(X[self.num_cols_])
        self.cat_imputer_ = SimpleImputer(strategy='most_frequent')
        self.cat_imputer_.fit(X[self.cat_cols_])
       
        # Fit encoder
        self.encoder_ = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_cat_imputed = self.cat_imputer_.transform(X[self.cat_cols_])
        self.encoder_.fit(X_cat_imputed)
       
        # Fit scaler
        self.scaler_ = StandardScaler()
        X_num_imputed = self.num_imputer_.transform(X[self.num_cols_])
        self.scaler_.fit(X_num_imputed)
        return self

    def transform(self, X):
        X = self._feature_engineering(X)
        X = X.drop(columns=self.cols_to_drop_)
        X = X.drop(columns=self.corr_cols_to_drop_)
        X_num = self.num_imputer_.transform(X[self.num_cols_])
        X_cat = self.cat_imputer_.transform(X[self.cat_cols_])
        X_cat_encoded = self.encoder_.transform(X_cat)
        X_num_scaled = self.scaler_.transform(X_num)
        X_final = np.hstack([X_num_scaled, X_cat_encoded])
        return X_final

# building a pipeline

In [18]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline(steps=[
    ('preprocessor', FraudPreprocessor(null_thresh=0.5, corr_thresh=0.9)),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor', FraudPreprocessor()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score

mlflow.set_experiment("LogisticRegression_Training")

experiments = [
    {"null_thresh": 0.5, "corr_thresh": 0.9, "solver": "lbfgs", "C": 1.0},
    {"null_thresh": 0.7, "corr_thresh": 0.9, "solver": "lbfgs", "C": 1.0},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "solver": "saga",  "C": 1.0},
    {"null_thresh": 0.5, "corr_thresh": 0.9, "solver": "lbfgs", "C": 0.1},
]

for exp in experiments:
    pipe = Pipeline(steps=[
        ('preprocessor', FraudPreprocessor(null_thresh=exp["null_thresh"], corr_thresh=exp["corr_thresh"])),
        ('model', LogisticRegression(
            solver=exp["solver"],
            C=exp["C"],
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))
    ])
    
    pipe.fit(X_train, y_train)
    
    with mlflow.start_run(run_name=f"LR_solver={exp['solver']}_C={exp['C']}_null={exp['null_thresh']}"):
        mlflow.log_params({**exp, "max_iter": 1000, "class_weight": "balanced"})
        
        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)[:, 1]
        y_train_proba = pipe.predict_proba(X_train)[:, 1]
        
        train_roc_auc = roc_auc_score(y_train, y_train_proba)
        test_roc_auc = roc_auc_score(y_test, y_proba)
        mlflow.log_metric("train_roc_auc", train_roc_auc)
        mlflow.log_metric("test_roc_auc", test_roc_auc)
        
        report = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric_name}", value)
        
        cm = confusion_matrix(y_test, y_pred)
        mlflow.log_metric("tn", cm[0,0])
        mlflow.log_metric("fp", cm[0,1])
        mlflow.log_metric("fn", cm[1,0])
        mlflow.log_metric("tp", cm[1,1])
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        cv_scores = cross_val_score(pipe, X_train, y_train, cv=skf, scoring='roc_auc')
        mlflow.log_metric("cv_mean_roc_auc", cv_scores.mean())
        mlflow.log_metric("cv_std_roc_auc", cv_scores.std())
        
        mlflow.sklearn.log_model(pipe, artifact_path="logistic_regression_pipeline")
        
        print(f"solver={exp['solver']} C={exp['C']} null={exp['null_thresh']} → Train: {train_roc_auc:.4f} Test: {test_roc_auc:.4f} CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")